# Flickr30k Hybrid independent audit

Kiểm tra độc lập ba seed, artifact SHA-256, metric, aggregate và closeout.

## Kaggle input bắt buộc

1. `F30K_T1S1_Hybrid_Protocol`
2. `F30K_T1S1_Hybrid_42`
3. `F30K_T1S1_Hybrid_123`
4. `F30K_T1S1_Hybrid_2026`
5. `F30K_T1S1_Hybrid_Aggregate`
6. `F30K_T1S1_Hybrid_Closeout`

Runtime: **CPU / Accelerator None / Internet Off**. Notebook không đọc ảnh,
không đọc Held-out Test, không huấn luyện và không xuất ONNX/INT8.

## Kiểm toán và xuất artifact

In [1]:
import csv
import hashlib
import json
import math
import os
import statistics
from datetime import datetime, timezone
from pathlib import Path

EXPERIMENT_ID = "F30K_Hybrid_Audit"
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("KLCN_OUTPUT_ROOT", "/kaggle/working")) / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(chunk):
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def unique_named(name):
    matches = sorted(p for p in INPUT_ROOT.rglob(name) if p.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(p) for p in matches])
    return matches[0]

def write_json(path, payload):
    temporary = Path(path).with_suffix(Path(path).suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

protocol_path = unique_named("F30K_T1S1_Hybrid_Protocol_RESULT.json")
aggregate_path = unique_named("F30K_T1S1_Hybrid_Aggregate_RESULT.json")
closeout_path = unique_named("F30K_T1S1_Hybrid_Closeout_RESULT.json")
protocol = read_json(protocol_path)
aggregate = read_json(aggregate_path)
closeout = read_json(closeout_path)

assert protocol["status"] == "locked_validation_only"
assert protocol["test_evaluation_authorized"] is False
assert aggregate["status"] == "complete"
assert aggregate["decision"] == "STOP_VALIDATION_NO_GAIN"
assert aggregate["test_set_used"] is False
assert aggregate["test_evaluation_authorized"] is False
assert closeout["status"] == "complete"
assert closeout["conclusion"] == "STOP_VALIDATION_NO_GAIN"
assert closeout["onnx_or_int8_authorized_for_hybrid"] is False
assert closeout["existing_locked_deployment_model_unchanged"] is True

seed_results = []
seed_rows = []
for seed in (42, 123, 2026):
    result_path = unique_named(f"F30K_T1S1_Hybrid_{seed}_RESULT.json")
    result = read_json(result_path)
    assert result["status"] == "complete"
    assert result["seed"] == seed
    assert result["method"] == "Hybrid Sequence KD + Dense Mapped Logit-KD"
    assert result["test_set_used"] is False
    assert result["test_evaluation_authorized"] is False
    assert result["hybrid_protocol_sha256"] == sha256(protocol_path)

    for relative_name, metadata in result["artifacts"].items():
        artifact = result_path.parent / relative_name
        assert artifact.is_file(), artifact
        assert artifact.stat().st_size == metadata["bytes"], artifact
        assert sha256(artifact) == metadata["sha256"], artifact

    with (result_path.parent / "metrics.csv").open(
        "r", encoding="utf-8-sig", newline=""
    ) as handle:
        metric_rows = list(csv.DictReader(handle))
    assert metric_rows
    best = max(metric_rows, key=lambda row: float(row["CIDEr_D"]))
    assert int(float(best["epoch"])) == result["best_epoch"]
    for metric in ("ce_loss", "BLEU_1", "BLEU_4", "METEOR", "ROUGE_L", "CIDEr_D"):
        assert math.isclose(
            float(best[metric]),
            float(result["best_validation_metrics"][metric]),
            rel_tol=0.0,
            abs_tol=1e-12,
        ), (seed, metric)

    metrics = result["best_validation_metrics"]
    seed_rows.append({
        "seed": seed,
        "best_epoch": result["best_epoch"],
        "CE_raw": round(metrics["ce_loss"], 6),
        "BLEU_1_x100": round(metrics["BLEU_1"] * 100, 2),
        "BLEU_4_x100": round(metrics["BLEU_4"] * 100, 2),
        "METEOR_x100": round(metrics["METEOR"] * 100, 2),
        "ROUGE_L_x100": round(metrics["ROUGE_L"] * 100, 2),
        "CIDEr_D_x100": round(metrics["CIDEr_D"] * 100, 2),
        "baseline_CIDEr_D_x100": round(result["baseline_validation_cider_d"] * 100, 2),
        "delta_CIDEr_D_x100": round(result["delta_validation_cider_d"] * 100, 2),
    })
    seed_results.append(result)

for metric in ("BLEU_1", "BLEU_4", "METEOR", "ROUGE_L", "CIDEr_D"):
    values = [r["best_validation_metrics"][metric] for r in seed_results]
    summary = aggregate["summary"][metric]
    assert math.isclose(statistics.mean(values), summary["kd_mean"], abs_tol=1e-12)
    assert math.isclose(statistics.stdev(values), summary["kd_sample_sd"], abs_tol=1e-12)

bootstrap = aggregate["combined_paired_bootstrap"]
assert bootstrap["replicates"] == 10000
assert bootstrap["ci95_upper"] < 0.0
assert bootstrap["bootstrap_positive_probability"] == 0.0
assert aggregate["consistent_negative_delta"] is True
assert aggregate["consistent_positive_delta"] is False

csv_path = WORK_ROOT / "hybrid_seed_metrics_x100.csv"
with csv_path.open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(seed_rows[0]))
    writer.writeheader()
    writer.writerows(seed_rows)

result_payload = {
    "schema_version": "f30k_hybrid_independent_audit_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "passed_closed_negative_result",
    "experiment_id": EXPERIMENT_ID,
    "seeds": [42, 123, 2026],
    "artifact_integrity": "PASSED",
    "metric_recomputation": "PASSED",
    "decision": "STOP_VALIDATION_NO_GAIN",
    "mean_CIDEr_D_x100": round(aggregate["summary"]["CIDEr_D"]["kd_mean"] * 100, 2),
    "mean_delta_CIDEr_D_x100": round(aggregate["summary"]["CIDEr_D"]["delta_mean"] * 100, 2),
    "combined_ci95_delta_CIDEr_D_x100": [
        round(bootstrap["ci95_lower"] * 100, 2),
        round(bootstrap["ci95_upper"] * 100, 2),
    ],
    "bootstrap_positive_probability": bootstrap["bootstrap_positive_probability"],
    "test_set_used": False,
    "test_evaluation_authorized": False,
    "onnx_or_int8_authorized": False,
    "deployment_model_unchanged": closeout["existing_locked_model"],
    "next_phase_authorized": "F30K_Final_Snapshot_V2",
    "source_sha256": {
        "protocol": sha256(protocol_path),
        "aggregate": sha256(aggregate_path),
        "closeout": sha256(closeout_path),
    },
}
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json(result_path, result_payload)

report_path = WORK_ROOT / "REPORT.md"
report_path.write_text(
    "# Flickr30k T1→S1 Hybrid KD — independent audit\n\n"
    "- Integrity: **PASSED**.\n"
    "- Three-seed mean CIDEr-D: "
    f"**{result_payload['mean_CIDEr_D_x100']:.2f}**.\n"
    "- Mean Δ CIDEr-D versus baseline: "
    f"**{result_payload['mean_delta_CIDEr_D_x100']:+.2f}**.\n"
    "- Combined paired-bootstrap 95% CI: "
    f"**[{result_payload['combined_ci95_delta_CIDEr_D_x100'][0]:+.2f}; "
    f"{result_payload['combined_ci95_delta_CIDEr_D_x100'][1]:+.2f}]**.\n"
    "- Decision: **STOP_VALIDATION_NO_GAIN**.\n"
    "- Held-out Test was not read; ONNX/INT8 is not authorized.\n",
    encoding="utf-8",
)

manifest_items = []
for artifact in (result_path, csv_path, report_path):
    manifest_items.append({
        "relative_path": artifact.name,
        "size_bytes": artifact.stat().st_size,
        "sha256": sha256(artifact),
    })
write_json(WORK_ROOT / "MANIFEST.json", {
    "schema_version": "f30k_hybrid_audit_manifest_v1",
    "experiment_id": EXPERIMENT_ID,
    "status": "complete",
    "artifacts": manifest_items,
    "test_set_used": False,
})

print("Audit status            :", result_payload["status"])
print("Mean CIDEr-D x100       :", result_payload["mean_CIDEr_D_x100"])
print("Mean delta x100         :", result_payload["mean_delta_CIDEr_D_x100"])
print("95% CI x100             :", result_payload["combined_ci95_delta_CIDEr_D_x100"])
print("Decision                :", result_payload["decision"])
print("ONNX/INT8 authorized    :", result_payload["onnx_or_int8_authorized"])

Audit status            : passed_closed_negative_result
Mean CIDEr-D x100       : 38.31
Mean delta x100         : -9.24
95% CI x100             : [-11.53, -6.94]
Decision                : STOP_VALIDATION_NO_GAIN
ONNX/INT8 authorized    : False
